In [1]:
import pandas as pd

In [2]:

from joblib import Parallel, delayed
import warnings
warnings.filterwarnings("ignore")
import numpy as np 

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import xgboost
from xgboost import XGBClassifier
import tensorflow as tf
model = tf.keras.Sequential()
from sklearn.model_selection import RandomizedSearchCV
from pprint import pprint
from scikeras.wrappers import KerasClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [15]:
from sklearn.metrics import f1_score

In [4]:
pd.set_option('display.max_columns', None)

In [ ]:
# X_train = pd.read_csv("../data/X_train_scaled.csv")
# X_test = pd.read_csv("../data/X_test_scaled.csv")

# X_train_70 = pd.read_csv("../data/X_train_pca_70.csv")
# X_test_70 = pd.read_csv("../data/X_test_pca_70.csv")

# # X_train_90 = pd.read_csv("../data/X_train_pca_90.csv")
# # X_test_90 = pd.read_csv("../data/X_test_pca_90.csv")

# y_train = pd.read_csv("../data/y_train.csv")
# y_test = pd.read_csv("../data/y_test.csv")

In [19]:
X_train = pd.read_csv("../data/X_train_2.csv")
X_test = pd.read_csv("../data/X_test_2.csv")

y_train = pd.read_csv("../data/y_train_2.csv")
y_test = pd.read_csv("../data/y_test_2.csv")

In [5]:
X_train = pd.read_csv("../data/X_train_3.csv")
X_test = pd.read_csv("../data/X_test_3.csv")

y_train = pd.read_csv("../data/y_train_3.csv")
y_test = pd.read_csv("../data/y_test_3.csv")

In [6]:
y_train = y_train.values.ravel()
y_test = y_test.values.ravel()

In [6]:
# Creating a dictionary of datas

datasets = {
    "Scaled": (X_train, X_test),
    "PCA_70": (X_train_70, X_test_70),
    "PCA_90": (X_train_90, X_test_90)
}

In [7]:
# Parameter for random forest

param_grid_rf = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 10, 20, 30, 40],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2],
}

In [ ]:
# Applying random search on random forest

rf_results = {}

for name, (Xtr, Xte) in datasets.items():

    print(f"Running RandomForest tuning on {name}")

    rf = RandomForestClassifier(random_state=100)

    rf_random = RandomizedSearchCV(
        estimator=rf,
        param_distributions=param_grid_rf,
        n_iter=20,
        cv=3,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    rf_random.fit(Xtr, y_train)

    best_model = rf_random.best_estimator_

    pred = best_model.predict(Xte)

    acc = accuracy_score(y_test, pred)

    rf_results[name] = {
        "Best_Params": rf_random.best_params_,
        "Accuracy": acc
    }

Running RandomForest tuning on Scaled
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Running RandomForest tuning on PCA_70
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Running RandomForest tuning on PCA_90
Fitting 3 folds for each of 20 candidates, totalling 60 fits


In [ ]:
# Viewing results of random forest

for dataset, info in rf_results.items():
    print("\nDataset:", dataset)
    print("Accuracy:", info["Accuracy"])
    print("Best Params:")
    print(info["Best_Params"])


Dataset: Scaled
Accuracy: 0.9886053759252045
Best Params:
{'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 20}

Dataset: PCA_70
Accuracy: 0.7712310089598753
Best Params:
{'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': 30}

Dataset: PCA_90
Accuracy: 0.7828204129333852
Best Params:
{'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': 30}


In [ ]:
# Parameters for XgBoost

param_grid_xgb = {
    "n_estimators": [100, 200, 300, 400],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1],
    "colsample_bytree": [0.7, 0.8, 0.9, 1],
    "gamma": [0, 0.1, 0.3],
    "min_child_weight": [1, 3, 5]
}

In [ ]:
# Training model

xgb_results = {}

for name, (Xtr, Xte) in datasets.items():

    print(f"\nRunning XGBoost tuning on {name}")

    xgb = XGBClassifier(
        objective="multi:softmax",
        num_class=4,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    )

    xgb_random = RandomizedSearchCV(
        estimator=xgb,
        param_distributions=param_grid_xgb,
        n_iter=20,
        cv=3,
        verbose=1,
        random_state=42,
        n_jobs=-1
    )

    xgb_random.fit(Xtr, y_train)

    best_model = xgb_random.best_estimator_

    pred = best_model.predict(Xte)

    acc = accuracy_score(y_test, pred)

    xgb_results[name] = {
        "Best_Params": xgb_random.best_params_,
        "Accuracy": acc
    }


Running XGBoost tuning on Scaled
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Running XGBoost tuning on PCA_70
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Running XGBoost tuning on PCA_90
Fitting 3 folds for each of 20 candidates, totalling 60 fits


In [ ]:
# Viewing results

pprint(xgb_results)

{'PCA_70': {'Accuracy': 0.7962602259446825,
            'Best_Params': {'colsample_bytree': 0.8,
                            'gamma': 0.3,
                            'learning_rate': 0.1,
                            'max_depth': 5,
                            'min_child_weight': 1,
                            'n_estimators': 400,
                            'subsample': 0.9}},
 'PCA_90': {'Accuracy': 0.8621932216595247,
            'Best_Params': {'colsample_bytree': 0.7,
                            'gamma': 0,
                            'learning_rate': 0.2,
                            'max_depth': 5,
                            'min_child_weight': 5,
                            'n_estimators': 400,
                            'subsample': 0.7}},
 'Scaled': {'Accuracy': 0.9953252824308532,
            'Best_Params': {'colsample_bytree': 1,
                            'gamma': 0.3,
                            'learning_rate': 0.01,
                            'max_depth': 3,
        

In [22]:
# Making a function to automate like randomsearch

def ANN_random_search(meta, neurons1=64, neurons2=32, learning_rate=0.001):

    n_features = meta["n_features_in_"]

    model = tf.keras.Sequential([
        tf.keras.layers.Dense(neurons1, activation="relu", input_shape=(n_features,)),
        tf.keras.layers.Dense(neurons2, activation="relu"),
        tf.keras.layers.Dense(4, activation="softmax")
    ])

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [23]:
ann = KerasClassifier(
    model=ANN_random_search,
    verbose=0
)

In [29]:
param_grid_ann = {
    "model__neurons1": [64, 128, 256, 512],
    "model__neurons2": [16, 32, 64, 128],
    "model__learning_rate": [0.001, 0.01],
    "batch_size": [32, 64],
    "epochs": [100, 200, 300, 400, 500]
}

In [25]:
ann_results = {}

for name, (Xtr, Xte) in datasets.items():

    print(f"\nRunning ANN tuning on {name}")

    ann_random = RandomizedSearchCV(
        estimator=ann,
        param_distributions=param_grid_ann,
        n_iter=10,
        cv=3,
        random_state=42,
        verbose=1,
        n_jobs=-1
    )

    ann_random.fit(Xtr, y_train)

    best_model = ann_random.best_estimator_

    pred = best_model.predict(Xte)

    acc = accuracy_score(y_test, pred)

    ann_results[name] = {
        "Best_Params": ann_random.best_params_,
        "Accuracy": acc
    }


Running ANN tuning on Scaled
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Running ANN tuning on PCA_70
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Running ANN tuning on PCA_90
Fitting 3 folds for each of 10 candidates, totalling 30 fits


In [26]:
for dataset, result in ann_results.items():
    print("\nDataset:", dataset)
    print("Accuracy:", result["Accuracy"])
    print("Best Params:", result["Best_Params"])


Dataset: Scaled
Accuracy: 0.9517919750681729
Best Params: {'model__neurons2': 16, 'model__neurons1': 128, 'model__learning_rate': 0.001, 'epochs': 100, 'batch_size': 64}

Dataset: PCA_70
Accuracy: 0.8648227502921698
Best Params: {'model__neurons2': 32, 'model__neurons1': 128, 'model__learning_rate': 0.01, 'epochs': 100, 'batch_size': 32}

Dataset: PCA_90
Accuracy: 0.9158550837553564
Best Params: {'model__neurons2': 64, 'model__neurons1': 64, 'model__learning_rate': 0.01, 'epochs': 100, 'batch_size': 64}


In [30]:
ann_random = RandomizedSearchCV(
    estimator=ann,
    param_distributions=param_grid_ann,
    n_iter=10,
    cv=3,
    random_state=42,
    n_jobs=-1
)

ann_random.fit(X_train, y_train)

print("Best Params:", ann_random.best_params_)

best_ann = ann_random.best_estimator_

pred = best_ann.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

Best Params: {'model__neurons2': 16, 'model__neurons1': 64, 'model__learning_rate': 0.001, 'epochs': 300, 'batch_size': 64}
Accuracy: 0.9727308141799766


In [ ]:
# Random Forest
# Dataset: Scaled
# Accuracy: 0.9886053759252045
# Best Params:
# {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 20}

# XgBoost
# 'Scaled': {'Accuracy': 0.9953252824308532,
#             'Best_Params': {'colsample_bytree': 1,
#                             'gamma': 0.3,
#                             'learning_rate': 0.01,
#                             'max_depth': 3,
#                             'min_child_weight': 3,
#                             'n_estimators': 200,
#                             'subsample': 1}}}

# Dataset: Scaled

#ANN
# Accuracy: 0.9517919750681729
# Best Params: {'model__neurons2': 16, 'model__neurons1': 128, 'model__learning_rate': 0.001, 'epochs': 100, 'batch_size': 64}

In [36]:
rf_best = RandomForestClassifier(
    n_estimators=500,
    min_samples_split=5,
    min_samples_leaf=1,
    max_depth=20,
    random_state=42
)

rf_best.fit(X_train, y_train)

rf_pred = rf_best.predict(X_test)

print("Random Forest Confusion Matrix")
print(confusion_matrix(y_test, rf_pred))

print("\nRandom Forest Classification Report")
print(classification_report(y_test, rf_pred))

Random Forest Confusion Matrix
[[1145   12    4    0]
 [   0 6440    0    0]
 [  90   10 1390    1]
 [   0    0    3 1173]]

Random Forest Classification Report
              precision    recall  f1-score   support

           0       0.93      0.99      0.96      1161
           1       1.00      1.00      1.00      6440
           2       0.99      0.93      0.96      1491
           3       1.00      1.00      1.00      1176

    accuracy                           0.99     10268
   macro avg       0.98      0.98      0.98     10268
weighted avg       0.99      0.99      0.99     10268



In [ ]:

xgb_best = XGBClassifier(
    colsample_bytree=1,
    gamma=0.3,
    learning_rate=0.01,
    max_depth=3,
    min_child_weight=3,
    n_estimators=200,
    subsample=1,
    random_state=42,
    objective="multi:softmax",
    num_class=4
)

xgb_best.fit(X_train, y_train)

xgb_pred = xgb_best.predict(X_test)

print("XGBoost Confusion Matrix")
print(confusion_matrix(y_test, xgb_pred))

print("\nXGBoost Classification Report")
print(classification_report(y_test, xgb_pred))

XGBoost Confusion Matrix
[[1115    0   46    0]
 [   0 6440    0    0]
 [   1    0 1489    1]
 [   0    0    0 1176]]

XGBoost Classification Report
              precision    recall  f1-score   support

           0       1.00      0.96      0.98      1161
           1       1.00      1.00      1.00      6440
           2       0.97      1.00      0.98      1491
           3       1.00      1.00      1.00      1176

    accuracy                           1.00     10268
   macro avg       0.99      0.99      0.99     10268
weighted avg       1.00      1.00      1.00     10268



In [9]:
importance = xgb_best.feature_importances_

# Combine feature names with importance
importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": importance
})

# Sort by importance
importance_df = importance_df.sort_values(by="Importance", ascending=False)

print("\nFeature Importance (Top 20)")
print(importance_df.head(20))


Feature Importance (Top 20)
   Feature  Importance
51      51    0.986801
33      33    0.001696
38      38    0.001461
30      30    0.001371
68      68    0.001335
88      88    0.001304
8        8    0.001232
76      76    0.001231
36      36    0.001212
0        0    0.001193
43      43    0.001164
2        2    0.000000
12      12    0.000000
13      13    0.000000
14      14    0.000000
15      15    0.000000
16      16    0.000000
17      17    0.000000
18      18    0.000000
1        1    0.000000


In [42]:
ann_model = tf.keras.Sequential([
    tf.keras.layers.Dense(256, activation="relu", input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(4, activation="softmax")
])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

ann_model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

ann_model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=64,
    verbose=0
)

ann_pred = ann_model.predict(X_test)
ann_pred = np.argmax(ann_pred, axis=1)

print("ANN Confusion Matrix")
print(confusion_matrix(y_test, ann_pred))

print("\nANN Classification Report")
print(classification_report(y_test, ann_pred))

321/321 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
ANN Confusion Matrix
[[1102   24   35    0]
 [  22 6346   72    0]
 [  54  133 1238   66]
 [   0    0   34 1142]]

ANN Classification Report
              precision    recall  f1-score   support

           0       0.94      0.95      0.94      1161
           1       0.98      0.99      0.98      6440
           2       0.90      0.83      0.86      1491
           3       0.95      0.97      0.96      1176

    accuracy                           0.96     10268
   macro avg       0.94      0.93      0.94     10268
weighted avg       0.96      0.96      0.96     10268



KeyError: 38

In [9]:
param_grid_xgb = {
    "n_estimators": [100, 200, 300, 400,500],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1],
    "colsample_bytree": [0.7, 0.8, 0.9, 1],
    "gamma": [0, 0.1, 0.3],
    "min_child_weight": [1, 3, 5]
}

class_counts = np.bincount(y_train)
class_weights = class_counts.max() / class_counts
sam_wt = np.array([class_weights[i] for i in y_train])

In [11]:
xgb_results = {}

xgb = XGBClassifier(
    objective="multi:softprob",   
    num_class=4,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb_random = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid_xgb,
    n_iter=20,
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

xgb_random.fit(X_train, y_train, sample_weight= sam_wt)

best_model = xgb_random.best_estimator_

pred = best_model.predict(X_test)

acc = accuracy_score(y_test, pred)

xgb_results= {
    "Best_Params": xgb_random.best_params_,
    "Accuracy": acc
}

Fitting 3 folds for each of 20 candidates, totalling 60 fits


In [12]:
pprint(xgb_results)

{'Accuracy': 0.9954226723802103,
 'Best_Params': {'colsample_bytree': 0.9,
                 'gamma': 0,
                 'learning_rate': 0.01,
                 'max_depth': 4,
                 'min_child_weight': 1,
                 'n_estimators': 300,
                 'subsample': 0.9}}


In [ ]:
xgb_best = XGBClassifier(
    colsample_bytree=0.9,
    gamma=0,
    learning_rate=0.01,
    max_depth=4,
    min_child_weight=1,
    n_estimators=300,
    subsample=0.9,
    random_state=42,
    objective="multi:softprob",
    num_class=4
)

xgb_best.fit(X_train, y_train, sample_weight=sam_wt)

xgb_pred = xgb_best.predict(X_test)

print("XGBoost Confusion Matrix")
print(confusion_matrix(y_test, xgb_pred))

print("\nXGBoost Classification Report")
print(classification_report(y_test, xgb_pred))

XGBoost Confusion Matrix
[[1116    0   45    0]
 [   0 6440    0    0]
 [   1    0 1489    1]
 [   0    0    0 1176]]

XGBoost Classification Report
              precision    recall  f1-score   support

           0       1.00      0.96      0.98      1161
           1       1.00      1.00      1.00      6440
           2       0.97      1.00      0.98      1491
           3       1.00      1.00      1.00      1176

    accuracy                           1.00     10268
   macro avg       0.99      0.99      0.99     10268
weighted avg       1.00      1.00      1.00     10268



In [21]:
importance = xgb_best.feature_importances_

# Combine feature names with importance
importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": importance
})

# Sort by importance
importance_df = importance_df.sort_values(by="Importance", ascending=False)

print("\nFeature Importance (Top 30)")
print(importance_df.head(30))


Feature Importance (Top 30)
                       Feature  Importance
38                Credit_Score    0.529514
62               Age_Oldest_TL    0.134842
10                     num_std    0.041844
1         num_times_delinquent    0.025285
24                     enq_L3m    0.024384
68         credit_activity_gap    0.021391
12               num_std_12mts    0.018262
2    max_recent_level_of_deliq    0.018160
21       time_since_recent_enq    0.017333
66            recent_enq_ratio    0.015134
19                  PL_enq_L6m    0.009808
4              num_deliq_12mts    0.009774
36      pct_PL_enq_L6m_of_ever    0.009520
11                num_std_6mts    0.008408
7              max_deliq_12mts    0.007525
13       recent_level_of_deliq    0.006749
9            num_times_60p_dpd    0.004075
22                    enq_L12m    0.002999
63               Age_Newest_TL    0.002874
56                     Gold_TL    0.002844
28                      GENDER    0.002766
51          pct_tl_closed

In [19]:
best_model = xgb_random.best_estimator_

# Step 1: Probabilities
probs = best_model.predict_proba(X_test)

# Step 2: Binary target (high risk vs rest)
y_test_binary = (y_test == 3).astype(int)

# Step 3: Threshold tuning
from sklearn.metrics import f1_score
import numpy as np

thresholds = np.arange(0.1, 0.9, 0.05)

best_thresh = 0.5
best_score = 0

for t in thresholds:
    preds = (probs[:, 3] > t).astype(int)
    score = f1_score(y_test_binary, preds)

    if score > best_score:
        best_score = score
        best_thresh = t

print("Best Threshold:", best_thresh)

# Step 4: Final predictions
final_preds = (probs[:, 3] > best_thresh).astype(int)

Best Threshold: 0.15000000000000002


In [20]:
print(classification_report(y_test_binary, final_preds))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      9092
           1       1.00      1.00      1.00      1176

    accuracy                           1.00     10268
   macro avg       1.00      1.00      1.00     10268
weighted avg       1.00      1.00      1.00     10268

